# Tests du projet RAG, étape par étape
Lancez les cellules **de haut en bas**, une par une. Après un **Restart**, recommencez toujours par la cellule 0.

## 0. Préparation (à lancer en premier, à chaque fois)

In [ ]:
import os, sys
# Se placer à la racine du projet, même après un Restart
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("Dossier de travail :", os.getcwd())

## 1. Séparer le texte source en un fichier par contrat
Mettez vos fichiers `.txt` (qui contiennent plusieurs contrats) dans `textes_source/`. Cette cellule crée `contrats/contrat_01.txt`, `contrat_02.txt`, etc. À lancer seulement quand le texte source change.

In [ ]:
import re
from pathlib import Path

Path("contrats").mkdir(exist_ok=True)
# Reconnaît "Contrat n° 5", "## Contrat n° 5", "**Contrat n° 5**"...
motif = re.compile(r"(?=^[\s#*]*Contrat\s+n\s*[°º]?\s*\d+)", re.MULTILINE | re.IGNORECASE)

for source in Path("textes_source").glob("*.txt"):
    texte = source.read_text(encoding="utf-8")
    for partie in motif.split(texte):
        m = re.match(r"[\s#*]*Contrat\s+n\s*[°º]?\s*(\d+)", partie, re.IGNORECASE)
        if m:
            num = int(m.group(1))
            propre = partie.strip().lstrip("#* ")
            Path(f"contrats/contrat_{num:02d}.txt").write_text(propre, encoding="utf-8")
            print("Créé : contrat", num, "depuis", source.name)

print("Fichiers dans contrats :", len(list(Path("contrats").glob("*.txt"))))

Si la cellule ci-dessus crée **0 fichier**, affichez le début du texte source pour voir le format des titres :

In [ ]:
for source in Path("textes_source").glob("*.txt"):
    print("=====", source.name)
    print(source.read_text(encoding="utf-8")[:500])

## 2. Extraction : le texte de chaque contrat est-il bien lu ?
Chaque contrat doit avoir plusieurs centaines de caractères.

In [ ]:
from extraction import lire_dossier

contrats = lire_dossier("contrats")
for nom, texte in contrats.items():
    titre = texte.splitlines()[0] if texte else "(VIDE)"
    print(f"{len(texte):>6} caractères | {nom} | {titre}")

## 3. Découpage : les articles sont-ils reconnus ?
On doit voir `préambule`, puis `Article 1, 2, 3...`. Si tout est en `préambule`, le motif des articles est à adapter dans `src/decoupage.py`.

In [ ]:
from decoupage import decouper_contrat

nom, texte = next(iter(contrats.items()))
morceaux = decouper_contrat(nom, texte)
print(nom, ":", len(morceaux), "morceaux\n")
for m in morceaux:
    print("Article", m["metadata"]["article"], "|", m["texte"][:110])

## 4. Embeddings : Ollama transforme-t-il un texte en vecteur ?
Avec `bge-m3`, on doit voir `1024` puis 5 nombres.

In [ ]:
from indexation import embed

v = embed(["Remboursement anticipé"])[0]
print(len(v), v[:5])

## 5. Indexation : (re)construire la base vectorielle
Efface l'ancienne base puis indexe tous les contrats. À relancer après chaque modification des contrats ou du découpage.

In [ ]:
import chromadb
from indexation import get_collection, indexer_contrat

client = chromadb.PersistentClient(path="base_vectorielle")
try:
    client.delete_collection("contrats")   # repartir d'une base propre
except Exception:
    pass

collection = get_collection()
for nom, texte in contrats.items():
    print(nom, ":", indexer_contrat(nom, texte, collection), "morceaux indexés")
print("\nTotal dans la base :", collection.count(), "morceaux")

## 6. Recherche : quels articles sont trouvés pour une question ?
Vérifiez que la bonne réponse se trouve dans au moins un des extraits.

In [ ]:
from rag import rechercher

question = "Quelle est l'indemnité de remboursement anticipé du rachat de crédit immobilier ?"
for p in rechercher(question):
    print(p["contrat"], "| article", p["article"])
    print("  ", p["texte"][:250], "\n")

## 7. Réponse complète du RAG (recherche + Ollama)

In [ ]:
from rag import repondre

resultat = repondre(question)
print(resultat["reponse"])
print("\nSources :", resultat["sources"])

Pour chercher **dans un seul contrat**, ajoutez son nom :

In [ ]:
resultat = repondre("Quel est le TEG ?", contrat="contrat_05")
print(resultat["reponse"])

## 8. Extraction des informations clés en JSON

In [ ]:
from rag import extraire_infos
import json

infos = extraire_infos(contrats["contrat_05"] if "contrat_05" in contrats else texte)
print(json.dumps(infos, indent=2, ensure_ascii=False))